In [4]:
import pandas as pd
import bs4 as bs
import re
from group import RateGroup, RateAgreement, RateSteps
import json

In [5]:
classDf = pd.read_pickle('classDf.pkl')
idContentDf = pd.read_pickle('idContentDf.pkl')

In [119]:
def getRateTables(tbsId, appendixText):
    ac = idContentDf.loc[idContentDf['ids'] == tbsId, 'htmlContent'].values[0].decode('utf-8', 'ignore')

    ac = ac.replace('“', '')
    ac = ac.replace('”', '')
    ac = ac.replace(u'\xa0', ' ')

    soup = bs.BeautifulSoup(ac,'lxml')

    tables = soup.select('h2:contains("%s") ~ table' %(appendixText))

    tables = tables + soup.select('h2:contains("%s") ~ section table' %(appendixText))

    return tables

In [367]:
def parseHTMLToJSON(tbsId, rateTables, sep):
    rates = []
    for rateCat in rateTables:
        #print(rateCat)
        steps = rateCat.select('thead th')

        if len(steps) == 0:
            #print('no steps found')
            steps = rateCat.select('table:not(tfoot) > tr')[0].select('th')

        #print(steps)

        agreementTable = rateCat.select('tbody tr')
        if len(agreementTable) == 0:
            #print('no tbody found')
            agreementTable = rateCat.select('table:not(tfoot) > tr')
            agreementTable.pop(0)

        #print(agreementTable)
        #print(len(agreementTable))
        rateAgreements = getAgreements(agreementTable, steps) #might be able to detect if tbody is present here and remove column 1 removing the need for step + 1

        grpLvl = re.split(' annual', rateCat.find('caption').getText(), flags=re.IGNORECASE)
        grp = grpLvl[0].split(sep)[0].strip()
        lvl = grpLvl[0].split(sep)[1].strip().lstrip('0').lstrip(sep).split('/')[0]
        print(grp + '-' + lvl)
        rateGrp = RateGroup(
            grp,
            lvl,       
            rateAgreements
        )
        rates.append(rateGrp)

    results = [rateGrp.to_dict() for rateGrp in rates]
    with open('data/' + tbsId + '.json', 'w') as json_file:
        json.dump(results, json_file)
    return results

In [259]:
def getAgreements(agreementTable, steps):
    rateAgreements = []
    for agreement in agreementTable:
        rateStepsList = []
        amounts = agreement.findAll('td') #all the cells inside each row - since the row names are th we can get amts only with td
        for idx,amount in enumerate(amounts):
            step = re.search('Step.[0-9]*', steps[idx+1].getText())
            if amount.find(' to ') == -1:
                newAmts = [amountgetText().replace(',', '')]
            else:
                newAmts = amount.getText().replace(',', '').split(' to ')
            agStep = int(re.search('[0-9]',step[0])[0])
            agAmts = []
            for amt in newAmts:
                if amt.isdigit():
                    agAmts.append(int(amt))
            if len(agAmts) > 0:
                rateStep = RateSteps(agStep, agAmts)
                rateStepsList.append(rateStep)

        rateAgreement = RateAgreement(agreement.find('time').get('datetime'), rateStepsList)
        rateAgreements.append(rateAgreement)

    return rateAgreements

In [8]:
def getJSON(tbsId, appendixStr, sep):
    rateTables = getRateTables(tbsId, appendixStr)
    return parseHTMLToJSON(tbsId, rateTables, sep)

In [307]:
page_1 = getJSON('1', '**Appendix A', ':')
page_3 = getJSON('3', '**Appendix A', ':')
page_4 = getJSON('4', '**Appendix A', ' - ')
page_5 = getJSON('5', '**Appendix A', ':')
page_7 = getJSON('7', '** Appendix A', ':')
page_9 = getJSON('9', '**Appendix B-1', ':')
page_10 = getJSON('10', '** Appendix A', ':')
page_11 = getJSON('11', '**Appendix A', ' - ') #kinda works
page_12 = getJSON('12', '**Appendix A', ':')
page_14 = getJSON('14', '**Appendix A', ':') #kinda works
page_18 = getJSON('18', '**Appendix A', ':')
page_26 = getJSON('26', '**Appendix A', ' - ')
page_27 = getJSON('27', '**Appendix A', ':')

In [226]:
with open('data/combined/payscales.json', 'w') as json_file:
        json.dump(combined_list, json_file)

In [368]:
page_11 = getJSON('1', '**Appendix A', ':')

CS-01-
CS-02-
CS-03-
CS-04-
CS-05-
